# Construção do dataset ISIC

Converte o dataset ISIC 2019 para o formato `SimpleLesionData` consumido pelo restante do pipeline (fine_tune.ipynb, analyze_tests.ipynb, etc.).


Saída gerada:
- `data/stt_data/images/` — imagens processadas
- `data/stt_data/simple_dataset.json` — dataset completo
- `data/stt_data/training_dataset.json` — split de treino (80%)
- `data/stt_data/test_dataset.json` — split de teste (20%)
- `data/simple_dataset_analysis.json` — análise do dataset

### Imports

In [ ]:
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.notebook import tqdm
from os.path import join, exists
from os import makedirs
from json import dump
from scripts.isic import (
    ISIC_2019_LABEL_COLUMNS,
    resolve_isic_2019_label,
    isic_2019_to_simple_lesion_data,
)

import pandas as pd
import scripts.definitions as defs
import scripts.data as dt

### Configuração

In [ ]:
ISIC_VERSION = '2019'
ISIC_PATH = join('..', 'data', 'isic_raw')
IMAGES_PATH  = join(ISIC_PATH, 'ISIC_2019_Training_Input')
GT_CSV_PATH  = join(ISIC_PATH, 'ISIC_2019_Training_GroundTruth.csv')
META_CSV_PATH = join(ISIC_PATH, 'ISIC_2019_Training_Metadata.csv')

### Carregamento do CSV e conversão para SimpleLesionData

In [ ]:
df = pd.read_csv(GT_CSV_PATH)
print(f'Linhas no CSV: {len(df)}')
print(f'Colunas: {list(df.columns)}')

dataset: list[dt.SimpleLesionData] = []
skipped = 0

if ISIC_VERSION == '2019':
    # Carrega metadata para obter lesion_id (agrupa imagens da mesma lesão)
    lesion_id_map: dict[str, str] = {}

    if META_CSV_PATH and exists(META_CSV_PATH):
        meta_df = pd.read_csv(META_CSV_PATH)

        if 'lesion_id' in meta_df.columns:
            lesion_id_map = dict(zip(meta_df['image'], meta_df['lesion_id']))
            print(f'Metadados carregados: {len(lesion_id_map)} entradas com lesion_id')
        else:
            print('Metadata CSV sem coluna lesion_id — usando image_name como grupo.')
    else:
        print('Metadata CSV não encontrado — usando image_name como grupo.')

    # Converte cada linha do ground truth
    # exam_id representa a lesão (ou imagem se não houver lesion_id)
    lesion_to_exam_id: dict[str, int] = {}
    next_id = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Convertendo ISIC 2019: '):
        row_dict = row.to_dict()
        image_name = str(row_dict['image'])

        label_code = resolve_isic_2019_label(row_dict)

        if label_code is None:
            skipped += 1
            continue

        # Usa lesion_id como grupo; cai no image_name se indisponível
        group_key = lesion_id_map.get(image_name, image_name)

        if group_key not in lesion_to_exam_id:
            lesion_to_exam_id[group_key] = next_id
            next_id += 1

        exam_id = lesion_to_exam_id[group_key]
        entry = isic_2019_to_simple_lesion_data(image_name, label_code, exam_id)

        if entry is not None:
            dataset.append(entry)
        else:
            skipped += 1

print(f'\nEntradas convertidas: {len(dataset)}')
print(f'Entradas descartadas (sem mapeamento): {skipped}')

### Processamento das imagens

Mesmo pré-processamento aplicado em `build_dataset.ipynb`:
conversão para RGB, redimensionamento para no máximo `MAX_IMAGE_SIZE` px
no maior lado, salvo como JPEG qualidade 100.

In [ ]:
output_images_dir = join(defs.DATA_PATH, 'stt_data', 'images')
makedirs(output_images_dir, exist_ok=True)

missing_images: list[str] = []

for entry in tqdm(dataset, desc='Processando imagens: '):
    src_path = join(IMAGES_PATH, entry.image)

    if not exists(src_path):
        # Tenta extensão .JPG maiúscula
        src_path_upper = src_path.replace('.jpg', '.JPG')
        if exists(src_path_upper):
            src_path = src_path_upper
        else:
            missing_images.append(entry.image)
            continue

    image = Image.open(src_path).convert('RGB')

    largest_side = max(image.size)

    if largest_side > defs.MAX_IMAGE_SIZE:
        scale = defs.MAX_IMAGE_SIZE / largest_side
        new_size = tuple(int(d * scale) for d in image.size)
        image = image.resize(new_size)  

    image.save(join(output_images_dir, entry.image), format='JPEG', quality=100)

if missing_images:
    print(f'\nImagens não encontradas: {len(missing_images)}')
    print('Primeiras 10:', missing_images[:10])

    # Remove do dataset entradas sem imagem
    missing_set = set(missing_images)
    dataset = [e for e in dataset if e.image not in missing_set]
    print(f'Dataset após remoção de imagens ausentes: {len(dataset)}')

print(f'\nImagens processadas: {len(dataset)}')

### Remoção de classes raras

Mesma regra do `build_dataset.ipynb`: classes com menos de 15 amostras são descartadas.

In [ ]:
# Análise prévia para identificar classes com poucas amostras
pre_analysis = dt.analyse_simple_dataset(dataset, defs.DATA_PATH, 'simple_dataset.json', save=False)

print('Distribuição antes da filtragem:')
for cls, info in pre_analysis.skin_lesion_distribution.classes.items():
    flag = '  [REMOVIDA]' if info.count < 15 else ''
    print(f'  {cls}: {info.count}{flag}')

dataset = [
    entry for entry in dataset
    if pre_analysis.skin_lesion_distribution.classes[entry.report.skin_lesion].count >= 15
]

print(f'\nDataset após filtragem: {len(dataset)} entradas')

### Análise e salvamento do dataset completo

In [ ]:
makedirs(join(defs.DATA_PATH, 'stt_data'), exist_ok=True)

dataset_name = 'simple_dataset.json'
dataset_path = join(defs.DATA_PATH, 'stt_data', dataset_name)

with open(dataset_path, 'w', encoding='utf-8') as file:
    dump([entry.model_dump() for entry in dataset], file, indent=4, ensure_ascii=False)

dataset_analysis = dt.analyse_simple_dataset(dataset, defs.DATA_PATH, dataset_name)

dt.plot_skin_lesion_distribution(dataset_analysis)
dt.plot_risk_distribution(dataset_analysis)

### Seccionamento treino / teste

Mesma estratégia do `build_dataset.ipynb`:
`StratifiedGroupKFold` com estratificação por `skin_lesion`
e agrupamento por `exam_id` (para evitar vazamento de dados
entre split de treino e teste quando há múltiplas imagens da mesma lesão ou paciente).

In [ ]:
exams  = [entry.exam_id for entry in dataset]
labels = [entry.report.skin_lesion for entry in dataset]

n_splits = int(1.0 / defs.TEST_PROPORTION)  # 5 para TEST_PROPORTION=0.2

stratified_group_kfold = StratifiedGroupKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=defs.STATIC_RANDOM_STATE
)

training_indices, test_indices = next(
    stratified_group_kfold.split(dataset, labels, exams)
)

training_data = [dataset[i] for i in training_indices]
test_data     = [dataset[i] for i in test_indices]

print(f'Treino: {len(training_data)} entradas')
print(f'Teste:  {len(test_data)} entradas')

# Salva e analisa cada split
splits = [
    (training_data, 'training_dataset.json'),
    (test_data,     'test_dataset.json'),
]

for split_dataset, split_name in splits:
    split_path = join(defs.DATA_PATH, 'stt_data', split_name)

    with open(split_path, 'w', encoding='utf-8') as file:
        dump([entry.model_dump() for entry in split_dataset], file, indent=4, ensure_ascii=False)

    split_analysis = dt.analyse_simple_dataset(split_dataset, defs.DATA_PATH, split_name)

    print(f'\n{split_name}:')
    for cls, info in split_analysis.skin_lesion_distribution.classes.items():
        print(f'  {cls}: {info.count} ({info.proportion:.1f}%)')

    dt.plot_skin_lesion_distribution(split_analysis)
    dt.plot_risk_distribution(split_analysis)

### Balanceamento do dataset de treino

O ISIC 2019 é severamente desbalanceado (NV representa ~50% dos dados).

Estratégia V7:
- **Cap em 3710** (2ª maior classe = MEL): reduz NV ao nível do MEL — mesma lógica do V2 que alcançou 77.6%
- **Floor em 1000**: oversampling moderado para classes pequenas (AK, DF, VASC) — dobro do V2 (500) sem o excesso do V5/V6 (2000)

O split de teste **não** é alterado para garantir avaliação imparcial na distribuição real.

In [ ]:
import random
from collections import defaultdict

random.seed(defs.STATIC_RANDOM_STATE)

# Agrupa entradas de treino por classe
class_groups: dict[str, list] = defaultdict(list)
for entry in training_data:
    class_groups[entry.report.skin_lesion].append(entry)

# V7: cap=3710 (mesmo do V2), floor=1000 (moderado, sem oversampling extremo)
max_cap = 3710
min_floor = 1000
print(f'Cap máximo: {max_cap}')
print(f'Floor mínimo: {min_floor}')

balanced: list = []
for cls, entries in class_groups.items():
    n = len(entries)
    target = max(min(n, max_cap), min_floor)

    if n >= target:
        sampled = random.sample(entries, target)
    else:
        sampled = entries.copy()
        while len(sampled) < target:
            remaining = target - len(sampled)
            sampled += random.sample(entries, min(len(entries), remaining))
        sampled = sampled[:target]

    balanced.extend(sampled)
    print(f'  {cls}: {n} → {len(sampled)}')

random.shuffle(balanced)
training_data = balanced
print(f'\nDataset de treino balanceado: {len(training_data)} entradas')

# Sobrescreve training_dataset.json com dados balanceados
balanced_path = join(defs.DATA_PATH, 'stt_data', 'training_dataset.json')
with open(balanced_path, 'w', encoding='utf-8') as file:
    dump([entry.model_dump() for entry in training_data], file, indent=4, ensure_ascii=False)

balanced_analysis = dt.analyse_simple_dataset(training_data, defs.DATA_PATH, 'training_dataset.json')
print('\nDistribuição balanceada:')
for cls, info in balanced_analysis.skin_lesion_distribution.classes.items():
    print(f'  {cls}: {info.count} ({info.proportion:.1f}%)')

dt.plot_skin_lesion_distribution(balanced_analysis)